## SACNN

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

In [2]:

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super(SelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        # x shape: (batch_size, seq_len, embed_dim)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Compute attention scores
        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (self.embed_dim ** 0.5)
        attn_probs = self.softmax(attn_scores)

        # Apply attention weights
        attn_output = torch.bmm(attn_probs, V)
        return attn_output

In [3]:

class SACNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SACNN, self).__init__()
        # EfficientNet-B0 pretrained backbone for feature extraction
        self.efficientnet = models.efficientnet_b0(pretrained=True)
        # Remove EfficientNet classifier to use as feature extractor
        self.efficientnet.classifier = nn.Identity()

        # 1D CNN layers to capture temporal dependencies
        self.conv1d = nn.Sequential(
            nn.Conv1d(in_channels=1280, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(512, 256, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # Self-Attention layer
        self.self_attention = SelfAttention(embed_dim=256)

        # Final classification layer
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # Input x shape: (batch_size, channels, height, width) - spectrogram images

        # Extract features with EfficientNet backbone (output shape: batch_size x 1280)
        features = self.efficientnet(x)

        # Add a temporal dimension to reshape for 1D CNN: (batch_size, channels=1280, seq_len=1)
        features = features.unsqueeze(-1)

        # 1D CNN to capture temporal features: output shape (batch_size, 256, seq_len=1)
        conv_out = self.conv1d(features)

        # Rearrange for attention: (batch_size, seq_len=1, embed_dim=256)
        conv_out = conv_out.transpose(1, 2)

        # Apply self-attention mechanism
        attn_out = self.self_attention(conv_out)

        # Remove sequence length dimension: (batch_size, 256)
        attn_out = attn_out.squeeze(1)

        # Classification output logits
        logits = self.fc(attn_out)

        return logits
